In [ ]:
import csv
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.preprocessing import StandardScaler


def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [37]:
# read data

df = pd.read_csv('src/pima-indians-diabetes.data', skiprows=2, header=None)

X = df.iloc[:, :-1] # everything except last column
y = df.iloc[:, -1] # last col

In [38]:

# using state=42 makes result reproducable. remove for random
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.4,
    random_state=42
)

# scale the values
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# verify shape 18-12
print(X_train.shape)
print(X_test.shape)

(460, 8)
(308, 8)


In [39]:
y_train = y_train.values
y_test = y_test.values

X_train = np.hstack((np.ones((X_train.shape[0], 1)), X_train))
X_test = np.hstack((np.ones((X_test.shape[0], 1)), X_test))

print(y_train.shape)
print(X_train.shape)

(460,)
(460, 9)


In [ ]:
# need a weight for all inputs BP Cholesterol Age Pregnant (w1,w2,w3,w4)
# need w0 (bias)
# score = w1*BP + w2*Cholesterol + w3*Age + w4*Pregnant + b

prev_loss = 0
current_loss = 0
threshold = 1e-6

fail_safe = 5000
iterations = 0
learning_rate = 0.01

epsilon = 1e-15 # const dont change

weights = np.zeros(X_train.shape[1])

while iterations < fail_safe:
    # 1. predict
    z = np.dot(X_train, weights)
    y_pred = sigmoid(z)

    # 2. calculate loss
    m = len(y_train)
    
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)

    current_loss = -(1/m) * np.sum(
        y_train * np.log(y_pred) + (1 - y_train) * np.log(1 - y_pred)
    )

    # 3. stop if converged
    if iterations > 0 and abs(current_loss - prev_loss) < threshold:
        print(f"Iterations: {iterations} | current_loss: {current_loss}")
        break

    # 4. check error
    error = y_pred - y_train

    # 5. adjust weight
    gradient = np.dot(X_train.T, error) / m
    weights = weights - learning_rate * gradient

    # 6. update for next loop
    prev_loss = current_loss
    iterations += 1


Iterations: 4375 | current_loss: 0.47648320824727025


In [41]:
# run the algo again but on test data
z_test = np.dot(X_test, weights)
y_prob = sigmoid(z_test)

# change probability into class label. Threshold is 0.5
y_pred = (y_prob >= 0.5).astype(int)

# compare accuracy
accuracy = np.mean(y_pred == y_test)

print(f"Accurary: {accuracy}")

Accurary: 0.7467532467532467


In [42]:
print("weights:")

for i, w in enumerate(weights):
    print(f"    w{i}: {w}")

weights:
    w0: -0.7363325527422436
    w1: 0.22935570484809484
    w2: 0.9217453080803271
    w3: -0.12554336575987898
    w4: -0.07351168714510992
    w5: -0.0021165583847268772
    w6: 0.7349435997798648
    w7: 0.14496170732924563
    w8: 0.3673401562692157
